In [1]:
pip install openai pandas

Note: you may need to restart the kernel to use updated packages.


# Baseline Setup

In [2]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    tie_word_embeddings=False
)

test_prompts = [
    "A store sells apples for $0.75 each and oranges for $1.25 each. If Sarah buys 4 apples and 3 oranges, how much does she spend in total?",
    "A train travels at 60 mph for 2.5 hours, then at 80 mph for 1.5 hours. What is the total distance traveled?",
    "A class has 32 students. 3/8 of them play football and 1/4 of them play basketball. How many students play neither sport?",
    "John earns $15 per hour and works 8 hours a day, 5 days a week. He saves 20% of his weekly earnings. How much does he save per week?",
    "A rectangle has a perimeter of 54 cm. If its length is twice its width, what is the area of the rectangle?",
    "A tank is 1/3 full of water. After adding 24 liters, it becomes 3/4 full. What is the total capacity of the tank?",
    "A shopkeeper bought 50 items for $200 and sold them all for $270. What is the profit percentage?",
    "Two friends split a restaurant bill. The bill was $84 before a 15% tip. They also have a coupon for $10 off the total after tip. How much does each person pay?",
    "A car depreciates in value by 15% each year. If it was worth $20,000 when new, what is it worth after 2 years?",
    "A pipe can fill a tank in 6 hours and another pipe can fill the same tank in 4 hours. If both pipes are opened together, how long will it take to fill the tank?"
]

print("Starting new baseline generation...")
responses = []

# 3. Generate answers
for i, prompt in enumerate(test_prompts):
    print(f"Processing Math Question {i+1}/10...")
    
    # Format for Qwen
    messages = [
        {"role": "system", "content": "You are a helpful mathematical reasoning assistant. Think step-by-step."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate (using max_new_tokens=2048 to prevent cutoff)
    outputs = model.generate(
        **inputs, 
        max_new_tokens=2048,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Extract only the generated text
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, outputs)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    responses.append(response)

# 4. Save to CSV
df_baseline = pd.DataFrame({
    'id': range(1, 11),
    'prompt': test_prompts,
    'model_response': responses
})

baseline_path = "/kaggle/working/math_baseline_responses.csv"
df_baseline.to_csv(baseline_path, index=False)
print(f"\nSuccess! New Math Baseline saved to {baseline_path}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Starting new baseline generation...
Processing Math Question 1/10...
Processing Math Question 2/10...
Processing Math Question 3/10...
Processing Math Question 4/10...
Processing Math Question 5/10...
Processing Math Question 6/10...
Processing Math Question 7/10...
Processing Math Question 8/10...
Processing Math Question 9/10...
Processing Math Question 10/10...

Success! New Math Baseline saved to /kaggle/working/math_baseline_responses.csv


# LLM as a Judge(Base)

In [3]:
import pandas as pd
import re
import time
from openai import OpenAI

# 1. Setup the API Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="" # Insert your Groq API key here
)

JUDGE_MODEL = "llama-3.3-70b-versatile" 

def evaluate_response(prompt, gold_response, model_response):
    judge_system_prompt = """
    You are an impartial expert evaluator. You will be provided with a mathematical reasoning prompt, a perfect 'Gold Standard' reference answer, and a 'Candidate Model' answer.
    
    Evaluate the candidate's answer based on:
    1. Logical accuracy.
    2. Step-by-step mathematical reasoning quality.
    3. Final answer correctness.
    
    Provide a brief explanation of your reasoning, and then output a final score out of 10 on a new line in this exact format: SCORE: [X]
    """
    user_content = f"PROMPT: {prompt}\n\nGOLD STANDARD RESPONSE: {gold_response}\n\nCANDIDATE MODEL RESPONSE: {model_response}"

    try:
        completion = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": judge_system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0 
        )
        evaluation = completion.choices[0].message.content
        match = re.search(r"SCORE:\s*([0-9]+(?:\.[0-9]+)?)", evaluation)
        score = float(match.group(1)) if match else None
        return score, evaluation
    except Exception as e:
        return None, str(e)


baseline_model_path = "/kaggle/working/math_baseline_responses.csv"
df = pd.read_csv(baseline_model_path)


gold_file_path = "/kaggle/input/datasets/abdullahkhalid03/evaluation-data-math/evaluation_data_math.csv" 
base_df = pd.read_csv(gold_file_path)[['id', 'gold_response']]


df = df.merge(base_df, left_on='id', right_on='id', how='left')


scores = []
explanations = []

print("Grading the RAW Base Model on Math Prompts...\n")
for index, row in df.iterrows():
    print(f"Grading Question {row['id']}/10...")
    score, eval_text = evaluate_response(row['prompt'], row['gold_response'], str(row['model_response']))
    scores.append(score)
    explanations.append(eval_text)
    time.sleep(1)


df['judge_score'] = scores
df['judge_explanation'] = explanations

output_path = "/kaggle/working/math_baseline_evaluated.csv"
df.to_csv(output_path, index=False)

print(f"\n Evaluation complete! Base Model Math Average Score: {df['judge_score'].mean()}/10")

Grading the RAW Base Model on Math Prompts...

Grading Question 1/10...
Grading Question 2/10...
Grading Question 3/10...
Grading Question 4/10...
Grading Question 5/10...
Grading Question 6/10...
Grading Question 7/10...
Grading Question 8/10...
Grading Question 9/10...
Grading Question 10/10...

✅ Evaluation complete! Base Model Math Average Score: 9.666666666666666/10


# LLM AS A JUDGE (SFT)

In [4]:
import pandas as pd
import re
import time
from openai import OpenAI

# 1. Setup the API Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="" # Insert your Groq API key here
)

JUDGE_MODEL = "llama-3.3-70b-versatile" 

def evaluate_response(prompt, gold_response, model_response):
    judge_system_prompt = """
    You are an impartial expert evaluator. You will be provided with a reasoning prompt, a perfect 'Gold Standard' reference answer, and a 'Candidate Model' answer.
    
    Evaluate the candidate's answer based on:
    1. Logical accuracy.
    2. Step-by-step mathematical reasoning quality.
    3. Final answer correctness.
    
    Provide a brief explanation of your reasoning, and then output a final score out of 10 on a new line in this exact format: SCORE: [X]
    """
    user_content = f"PROMPT: {prompt}\n\nGOLD STANDARD RESPONSE: {gold_response}\n\nCANDIDATE MODEL RESPONSE: {model_response}"

    try:
        completion = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": judge_system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0 
        )
        evaluation = completion.choices[0].message.content
        match = re.search(r"SCORE:\s*([0-9]+(?:\.[0-9]+)?)", evaluation)
        score = float(match.group(1)) if match else None
        return score, evaluation
    except Exception as e:
        return None, str(e)

sft_file_path = "/kaggle/input/datasets/abdullahkhalid03/sft-all-trials-3/sft_all_trial_responses-3.csv"
df = pd.read_csv(sft_file_path)

baseline_file_path = "/kaggle/input/datasets/abdullahkhalid03/evaluation-data-math/evaluation_data_math.csv" 

base_df = pd.read_csv(baseline_file_path)[['id', 'gold_response']]

df = df.merge(base_df, left_on='question_id', right_on='id', how='left')

scores = []
explanations = []

print(f"Starting evaluation of {len(df)} total math rows...")
for index, row in df.iterrows():
    print(f"Grading Trial {row['trial_id']}, Question {row['question_id']}...")
    score, eval_text = evaluate_response(row['prompt'], row['gold_response'], str(row['model_response']))
    scores.append(score)
    explanations.append(eval_text)
    time.sleep(1)

df['judge_score'] = scores
df['judge_explanation'] = explanations


print("\n --- MATH SFT LEADERBOARD --- ")
leaderboard = df.groupby('trial_id')['judge_score'].mean().sort_values(ascending=False)
for rank, (trial, score) in enumerate(leaderboard.items(), 1):
    print(f"{rank}. Trial {trial}: {score:.2f}/10")
    
output_path = "/kaggle/working/sft_math_trials_evaluated.csv"
df.to_csv(output_path, index=False)
print(f"\nDetailed evaluations saved to {output_path}")
print(f"Tell your GRPO lead that Trial {leaderboard.index[0]} is the official winner!")

Starting evaluation of 50 total math rows...
Grading Trial 1, Question 1...
Grading Trial 1, Question 2...
Grading Trial 1, Question 3...
Grading Trial 1, Question 4...
Grading Trial 1, Question 5...
Grading Trial 1, Question 6...
Grading Trial 1, Question 7...
Grading Trial 1, Question 8...
Grading Trial 1, Question 9...
Grading Trial 1, Question 10...
Grading Trial 2, Question 1...
Grading Trial 2, Question 2...
Grading Trial 2, Question 3...
Grading Trial 2, Question 4...
Grading Trial 2, Question 5...
Grading Trial 2, Question 6...
Grading Trial 2, Question 7...
Grading Trial 2, Question 8...
Grading Trial 2, Question 9...
Grading Trial 2, Question 10...
Grading Trial 3, Question 1...
Grading Trial 3, Question 2...
Grading Trial 3, Question 3...
Grading Trial 3, Question 4...
Grading Trial 3, Question 5...
Grading Trial 3, Question 6...
Grading Trial 3, Question 7...
Grading Trial 3, Question 8...
Grading Trial 3, Question 9...
Grading Trial 3, Question 10...
Grading Trial 4, Quest

# LLM as a Judge (GRPO sft trial 01)

In [4]:
import pandas as pd
import re
import time
from openai import OpenAI

# 1. Setup the API Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="" # Insert your API key here
)

JUDGE_MODEL = "llama-3.3-70b-versatile" 

def evaluate_response(prompt, gold_response, model_response):
    judge_system_prompt = """
    You are an impartial expert evaluator. You will be provided with a mathematical reasoning prompt, a perfect 'Gold Standard' reference answer, and a 'Candidate Model' answer.
    
    Evaluate the candidate's answer based on:
    1. Logical accuracy.
    2. Step-by-step mathematical reasoning quality.
    3. Final answer correctness.
    
    Provide a brief explanation of your reasoning, and then output a final score out of 10 on a new line in this exact format: SCORE: [X]
    """
    user_content = f"PROMPT: {prompt}\n\nGOLD STANDARD RESPONSE: {gold_response}\n\nCANDIDATE MODEL RESPONSE: {model_response}"

    try:
        completion = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": judge_system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0 
        )
        evaluation = completion.choices[0].message.content
        match = re.search(r"SCORE:\s*([0-9]+(?:\.[0-9]+)?)", evaluation)
        score = float(match.group(1)) if match else None
        return score, evaluation
    except Exception as e:
        return None, str(e)

gold_file_path = "/kaggle/input/datasets/abdullahkhalid03/evaluation-data-math/evaluation_data_math.csv" 
base_df = pd.read_csv(gold_file_path)[['id', 'gold_response']]

grpo_files = [
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trials-answers/grpo_trial_1_answers.csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trials-answers/grpo_trial_2_answers.csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trials-answers/grpo_trial_3_answers.csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trials-answers/grpo_trial_4_answers.csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trials-answers/grpo_trial_5_answers.csv"
]

final_leaderboard = {}

print("Starting Final GRPO Evaluation Phase...\n")


for i, file_path in enumerate(grpo_files):
    trial_name = f"GRPO Trial {i+1}"
    print(f"--- Grading {trial_name} ---")
    
    df = pd.read_csv(file_path)

    df = df.merge(base_df, on='id', how='left')
    
    scores = []
    explanations = []
    
    for index, row in df.iterrows():
        print(f"  Scoring Question {row['id']}...")
        score, eval_text = evaluate_response(row['prompt'], row['gold_response'], str(row['model_response']))
        scores.append(score)
        explanations.append(eval_text)
        time.sleep(1) # API Rate Limit Pause
        
    df['judge_score'] = scores
    df['judge_explanation'] = explanations
    
    output_filename = f"/kaggle/working/evaluated_grpo_trial_{i+1}.csv"
    df.to_csv(output_filename, index=False)
    
    avg_score = pd.Series(scores).mean()
    final_leaderboard[trial_name] = avg_score
    print(f" {trial_name} Complete! Average Score: {avg_score}/10\n")

print("\n --- FINAL GRPO LEADERBOARD --- ")
sorted_scores = sorted(final_leaderboard.items(), key=lambda item: item[1], reverse=True)
for rank, (trial, score) in enumerate(sorted_scores, 1):
    print(f"{rank}. {trial}: {score:.2f}/10")

print(f"\nAll files saved to /kaggle/working/. The ultimate champion is {sorted_scores[0][0]}!")

Starting Final GRPO Evaluation Phase...

--- Grading GRPO Trial 1 ---
  Scoring Question 1...
  Scoring Question 2...
  Scoring Question 3...
  Scoring Question 4...
  Scoring Question 5...
  Scoring Question 6...
  Scoring Question 7...
  Scoring Question 8...
  Scoring Question 9...
  Scoring Question 10...
✅ GRPO Trial 1 Complete! Average Score: 5.8/10

--- Grading GRPO Trial 2 ---
  Scoring Question 1...
  Scoring Question 2...
  Scoring Question 3...
  Scoring Question 4...
  Scoring Question 5...
  Scoring Question 6...
  Scoring Question 7...
  Scoring Question 8...
  Scoring Question 9...
  Scoring Question 10...
✅ GRPO Trial 2 Complete! Average Score: 6.3/10

--- Grading GRPO Trial 3 ---
  Scoring Question 1...
  Scoring Question 2...
  Scoring Question 3...
  Scoring Question 4...
  Scoring Question 5...
  Scoring Question 6...
  Scoring Question 7...
  Scoring Question 8...
  Scoring Question 9...
  Scoring Question 10...
✅ GRPO Trial 3 Complete! Average Score: 6.2/10

--- G

# LLM as a Judge (GRPO sft trial 01)

In [5]:
import pandas as pd
import re
import time
from openai import OpenAI

# 1. Setup the API Client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="" # Insert your API key here
)

JUDGE_MODEL = "llama-3.3-70b-versatile" 

def evaluate_response(prompt, gold_response, model_response):
    judge_system_prompt = """
    You are an impartial expert evaluator. You will be provided with a mathematical reasoning prompt, a perfect 'Gold Standard' reference answer, and a 'Candidate Model' answer.
    
    Evaluate the candidate's answer based on:
    1. Logical accuracy.
    2. Step-by-step mathematical reasoning quality.
    3. Final answer correctness.
    
    Provide a brief explanation of your reasoning, and then output a final score out of 10 on a new line in this exact format: SCORE: [X]
    """
    user_content = f"PROMPT: {prompt}\n\nGOLD STANDARD RESPONSE: {gold_response}\n\nCANDIDATE MODEL RESPONSE: {model_response}"

    try:
        completion = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": judge_system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0 
        )
        evaluation = completion.choices[0].message.content
        match = re.search(r"SCORE:\s*([0-9]+(?:\.[0-9]+)?)", evaluation)
        score = float(match.group(1)) if match else None
        return score, evaluation
    except Exception as e:
        return None, str(e)

gold_file_path = "/kaggle/input/datasets/abdullahkhalid03/evaluation-data-math/evaluation_data_math.csv" 
base_df = pd.read_csv(gold_file_path)[['id', 'gold_response']]

grpo_files = [
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trial01-answers/grpo_trial_1_answers (1).csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trial01-answers/grpo_trial_2_answers (1).csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trial01-answers/grpo_trial_3_answers (1).csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trial01-answers/grpo_trial_4_answers (1).csv",
    "/kaggle/input/datasets/abdullahkhalid03/grpo-trial01-answers/grpo_trial_5_answers (1).csv"
]

final_leaderboard = {}

print("Starting Final GRPO Evaluation Phase (SFT Trial 1 Base)...\n")

for i, file_path in enumerate(grpo_files):
    trial_name = f"GRPO Trial {i+1} (SFT-1 Base)"
    print(f"--- Grading {trial_name} ---")
    

    df = pd.read_csv(file_path)
    
    df = df.merge(base_df, on='id', how='left')
    
    scores = []
    explanations = []
    
    for index, row in df.iterrows():
        print(f"  Scoring Question {row['id']}...")
        score, eval_text = evaluate_response(row['prompt'], row['gold_response'], str(row['model_response']))
        scores.append(score)
        explanations.append(eval_text)
        time.sleep(1) # API Rate Limit Pause
        
    df['judge_score'] = scores
    df['judge_explanation'] = explanations
    
    output_filename = f"/kaggle/working/evaluated_grpo_sft1_trial_{i+1}.csv"
    df.to_csv(output_filename, index=False)
    
    avg_score = pd.Series(scores).mean()
    final_leaderboard[trial_name] = avg_score
    print(f" {trial_name} Complete! Average Score: {avg_score}/10\n")

print("\n --- FINAL GRPO LEADERBOARD (SFT 1 BASE) --- ")
sorted_scores = sorted(final_leaderboard.items(), key=lambda item: item[1], reverse=True)
for rank, (trial, score) in enumerate(sorted_scores, 1):
    print(f"{rank}. {trial}: {score:.2f}/10")

print(f"\nAll files saved to /kaggle/working/. The ultimate champion is {sorted_scores[0][0]}!")

Starting Final GRPO Evaluation Phase (SFT Trial 1 Base)...

--- Grading GRPO Trial 1 (SFT-1 Base) ---
  Scoring Question 1...
  Scoring Question 2...
  Scoring Question 3...
  Scoring Question 4...
  Scoring Question 5...
  Scoring Question 6...
  Scoring Question 7...
  Scoring Question 8...
  Scoring Question 9...
  Scoring Question 10...
✅ GRPO Trial 1 (SFT-1 Base) Complete! Average Score: 8.3/10

--- Grading GRPO Trial 2 (SFT-1 Base) ---
  Scoring Question 1...
  Scoring Question 2...
  Scoring Question 3...
  Scoring Question 4...
  Scoring Question 5...
  Scoring Question 6...
  Scoring Question 7...
  Scoring Question 8...
  Scoring Question 9...
  Scoring Question 10...
✅ GRPO Trial 2 (SFT-1 Base) Complete! Average Score: 8.8/10

--- Grading GRPO Trial 3 (SFT-1 Base) ---
  Scoring Question 1...
  Scoring Question 2...
  Scoring Question 3...
  Scoring Question 4...
  Scoring Question 5...
  Scoring Question 6...
  Scoring Question 7...
  Scoring Question 8...
  Scoring Question